In [ ]:
import json
import os
from dotenv import load_dotenv
load_dotenv()  # before project imports so ONTODISCO_LOG_LEVEL is applied correctly

from pymongo.mongo_client import MongoClient
import numpy as np

from src.ontodisco.utils.openai_utils import LLMTripletExtractor
from src.ontodisco.type_dedup import deduplicate_types

def get_mongo_client(mongo_uri):
    client = MongoClient(mongo_uri)
    return client

14:44:23 INFO     numexpr.utils — Note: detected 80 virtual cores but NumExpr set to maximum of 64, check "NUMEXPR_MAX_THREADS" environment variable.
14:44:23 INFO     numexpr.utils — Note: NumExpr detected 80 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
14:44:23 INFO     numexpr.utils — NumExpr defaulting to 16 threads.


In [2]:
client = get_mongo_client("mongodb://localhost:27018/?directConnection=true")

In [3]:
db = client.get_database("musique_gpt4_1_mini_onto_triplets")

types = []
for triplet in db.get_collection("initial_triplets").find({}, {"_id": 0, "subject_type": 1, "object_type": 1}):
    types.append(triplet["subject_type"])
    types.append(triplet["object_type"])
types = list(types)
len(types)

79014

In [ ]:
db = client.get_database("musique_gpt4_1_mini_onto_triplets")

triplets = []
for triplet in db.get_collection("initial_triplets").find({}, {"_id": 0}):
    triplets.append(triplet)
with open("musique_initial_triplets.jsonl", 'w') as f:
    for triplet in triplets:
        f.write(json.dumps(triplet) + "\n")

In [ ]:
verifier = LLMTripletExtractor(model="Openai/Gpt-oss-120b", api_key=os.getenv("AIRI_KEY"), base_url=os.getenv("AIRI_BASE_URL"))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

23:59:11 INFO     src.ontodisco.OpenAIUtils — Model: Openai/Gpt-oss-120b. Input price: 0.05. Output price: 0.2.


In [ ]:
deduplicate_types()

In [ ]:
with open("onto_artifacts/verified_groups.json", "r") as f:
    groups = json.load(f)

cluster_lens = []
for cluster in groups:
    cluster_name, members = cluster[0], cluster[1]
    cluster_lens.append(len(members))

cluster_lens = np.array(cluster_lens)
print(np.mean(cluster_lens), min(cluster_lens), max(cluster_lens))
len(cluster_lens)

In [ ]:
from collections import defaultdict

entity_type_mapping = defaultdict(list)
for cluster in groups:
    cluster_name, members = cluster[0], cluster[1]
    entity_type_mapping[cluster_name] = members
entity_type_mapping


# ------------- 

In [41]:
from src.ontodisco.relation_dedup import deduplicate_relations

all_relation_surface_forms = db.get_collection("initial_triplets").find({}, {"_id": 0, "subject_type": 1, "object_type": 1, "relation": 1})
all_relation_surface_forms = list(all_relation_surface_forms)

In [42]:
relation_dedup_result = deduplicate_relations(triplets=all_relation_surface_forms, llm_extractor=verifier, similarity_threshold=0.75)

00:10:59 INFO     src.ontodisco.relation_dedup — Collected 10318 relation subject types and 10318 relation object types from 39507 triplets
00:10:59 INFO     src.ontodisco.relation_dedup — Collected 39507 relation mentions from 39507 triplets


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

00:11:01 INFO     src.ontodisco.utils.dedup_base — After normalisation: 10316 unique labels (from 10318 raw labels)
00:11:49 INFO     src.ontodisco.relation_dedup — FAISS NN clustering: 10316 entities, top_k=50, threshold=0.75 → 18006 similar pairs → 5285 clusters
00:11:49 INFO     src.ontodisco.utils.dedup_base — Clustering produced 5285 clusters (1043 with 2+ members, requiring LLM verification)
00:11:49 INFO     src.ontodisco.utils.dedup_base — Mean cluster size for clusters with 2+ members: 5.8
LLM cluster verify:   0%|          | 1/5285 [00:01<2:18:31,  1.57s/cluster]00:13:13 WARNING  src.ontodisco.utils.dedup_base — LLM returned label 'first used' not matching any cluster 690 member ['abbreviation', 'abbreviation for', 'abbreviation of', 'abolished', 'abolished by', 'abolished in', 'abolished on', 'academic degree', 'academic degree received', 'acquired', 'acquired by', 'acquired shares in', 'acquired shares of', 'acquired status', 'acronym for', 'acted in', 'acted in role', 'act

Tokens spent: 44262 (prompt=20008, completion=24254)


LLM cluster verify:   0%|          | 18/5285 [02:13<4:36:46,  3.15s/cluster] 

...

Tokens spent: 386136 (prompt=164497, completion=221639)


LLM cluster verify:  40%|███▉      | 2096/5285 [29:42<21:38,  2.46cluster/s]

Tokens spent: 387236 (prompt=165032, completion=222204)


LLM cluster verify:  40%|███▉      | 2102/5285 [29:47<45:07,  1.18cluster/s]


KeyboardInterrupt: 

In [5]:
import json
with open("relation_dedup_result.json", 'r') as f:
    rels = json.load(f)

count = 0
for name, members in rels:
    if len(members) > 1:
        print(name, members)
        count += 1

abbreviation for ['abbreviation', 'abbreviation for', 'acronym for']
hasability ['ability', 'has ability']
abolished ['abolished', 'was abolished']
abolishedin ['abolished in', 'was abolished in']
abolishedon ['abolished on', 'were abolished on']
abolishesselfgovernmentof ['abolished self-government of', 'self-government abolished by']
rejects invitation from ['declined invitation from', 'declined invitations from', 'invitation rejected by', 'refused invitations from', 'rejected invitation from', 'rejected invitation of']
accessed via ['accessed through', 'accessed via']
accused_of ['accused', 'accused of']
achieved best world cup result as co-host ['achieved best World Cup result as co-host', 'achieved best world cup result as co-host']
acquired_by ['acquired', 'acquired by', 'bought by', 'purchased by', 'sold by']
acquire_shares ['acquired shares in', 'acquired shares of']
convicted_of ['convicted', 'convicted for', 'convicted of']
acquitted_of ['acquitted', 'was acquitted by']
acted

In [40]:
for item in relation_dedup_result.items.values():
    if len(item.surface_forms) > 1:
        print(item.canonical_label, item.surface_forms)

all-defensive first team honors ['All-Defensive First Team honors', 'all-defensive first team honors']
all-nba team selections ['All-NBA First Team designations', 'all-nba first team designations', 'number selected to All-NBA Team', 'number selected to all-nba team']
nba scoring champion ['led NBA in scoring', 'led nba in scoring', 'most recent NBA scoring champion', 'most recent nba scoring champion', 'youngest scoring leader in NBA history', 'youngest scoring leader in nba history']
nba all-star selections ['NBA All-Star', 'NBA All-Star Game appearances', 'NBA All-Star selections', 'nba all-star', 'nba all-star game appearances', 'nba all-star selections', 'number selected to All-Star Game', 'number selected to all-star game']
nba player averaging a triple-double in a season ['NBA player to average triple-double in a season', 'nba player to average triple-double in a season']
includes ['also includes', 'include', 'includes']
position held ['appointed to position', 'position', 'positi

In [39]:
item.surface_form_counts

{'academic degree': 1}

In [ ]:
import requests

# Получение списка моделей
response = requests.get('https://inference.airi.net:46783/v1/models', headers={"Authorization": "Bearer "+ os.getenv("AIRI_KEY")})
models = response.json()
models

{'object': 'list',
 'data': [{'id': 'AIRI/OCC-1-1.7B',
   'object': 'model',
   'created': 1784640466,
   'owned_by': 'vllm',
   'root': 's3://a030dylov/nlp-data/models/occ-ai/OCC-RAG-1.7B/733cc24406179feedbd4bf54b4723c8d414bbc94/7ab534b2-ddcd-47e6-a759-80375a7a2756',
   'parent': None,
   'max_model_len': 32768,
   'permission': [{'id': 'modelperm-b967fd7af4e327d4',
     'object': 'model_permission',
     'created': 1784640466,
     'allow_create_engine': False,
     'allow_sampling': True,
     'allow_logprobs': True,
     'allow_search_indices': False,
     'allow_view': True,
     'allow_fine_tuning': False,
     'organization': '*',
     'group': None,
     'is_blocking': False}]},
  {'id': 'Openai/Gpt-oss-120b',
   'object': 'model',
   'created': 1784640466,
   'owned_by': 'vllm',
   'root': '/models/Openai/Gpt-oss-120b',
   'parent': None,
   'max_model_len': 131072,
   'permission': [{'id': 'modelperm-ac0c60b0d8caeab5',
     'object': 'model_permission',
     'created': 178464